# Kaggle histopathology introduction
This notebook is an introduction to the data challenge of out of distribution classification of histopathology patches. It also serves as a baseline for the code and the model.

If you have any questions, feel free to contact me at [leo.fillioux@centralesupelec.fr](mailto:leo.fillioux@centralesupelec.fr).

In [ ]:
import h5py
import torch
import random
import numpy as np
import pandas as pd
import torchmetrics
import matplotlib.pyplot as plt
import torchvision.transforms as transforms
from tqdm.notebook import tqdm
from torch.utils.data import Dataset, DataLoader

import warnings

warnings.filterwarnings("ignore", category=UserWarning)

In [ ]:
TRAIN_IMAGES_PATH = "../data/train.h5"
VAL_IMAGES_PATH = "../data/val.h5"
TEST_IMAGES_PATH = "../data/test.h5"
SEED = 0

In [ ]:
torch.random.manual_seed(SEED)
random.seed(SEED)

## 1. Introduction to the data
The dataset consists of patches of whole slide images which should be classified into either containing tumor or not. The training images come from 3 different centers (i.e. hospitals), while the validation set comes from another center and the test set from yet another center. The visual aspect of the patches are quite different due to the slightly different staining procedures, conditions, and equipment from each hospital. The objective of the task is to build a classifier that is impacted by this distribution shift as little as possible.

The data is stored in `.h5` files, which can be seen as a folder hierarchy, which are can be seen as the following.
```
├── idx           # index of the image
│   └── img       # image in a tensor format
│   └── label     # binary label of the image
│   └── metadata  # some metadata on the images
```
The metadata is included for completeness but is not necessarily useful. The first element in the metadata corresponds to the center.

The following is a visualization of how different the images look from the different centers.

In [ ]:
train_images = {0: {0: None, 1: None}, 3: {0: None, 1: None}, 4: {0: None, 1: None}}
val_images = {1: {0: None, 1: None}}

In [ ]:
for img_data, data_path in zip(
    [train_images, val_images], [TRAIN_IMAGES_PATH, VAL_IMAGES_PATH]
):
    with h5py.File(data_path, "r") as hdf:
        for img_idx in list(hdf.keys()):
            label = int(np.array(hdf.get(img_idx).get("label")))
            center = int(np.array(hdf.get(img_idx).get("metadata"))[0])
            if img_data[center][label] is None:
                img_data[center][label] = np.array(hdf.get(img_idx).get("img"))
            if all(
                all(value is not None for value in inner_dict.values())
                for inner_dict in img_data.values()
            ):
                break
all_data = {**train_images, **val_images}

In [ ]:
fig, axs = plt.subplots(2, 4, figsize=(20, 10))
center_ids = {center: idx for idx, center in enumerate(all_data.keys())}
for center in all_data:
    for label in all_data[center]:
        axs[label, center_ids[center]].imshow(
            np.moveaxis(all_data[center][label], 0, -1).astype(np.float32)
        )
        axs[label, center_ids[center]].axis("off")
        if label == 0:
            axs[label, center_ids[center]].set_title(f"Center {center}")
plt.show()

## 2. Building a baseline model
The baseline model consists of extracting DINOv2 embeddings and linear probing.

In [ ]:
BATCH_SIZE = 16

### 2.1. Baseline dataset
We start by creating the model to read and process the data. For this simple model we also use another dataset with the preprocessed embeddings to avoid recomputing the same embeddings each time.

In [ ]:
class BaselineDataset(Dataset):
    def __init__(self, dataset_path, preprocessing, mode):
        super(BaselineDataset, self).__init__()
        self.dataset_path = dataset_path
        self.preprocessing = preprocessing
        self.mode = mode

        with h5py.File(self.dataset_path, "r") as hdf:
            self.image_ids = list(hdf.keys())

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        img_id = self.image_ids[idx]
        with h5py.File(self.dataset_path, "r") as hdf:
            img = torch.tensor(hdf.get(img_id).get("img"))
            label = (
                np.array(hdf.get(img_id).get("label")) if self.mode == "train" else None
            )
        return self.preprocessing(img).float(), label

In [ ]:
def precompute(dataloader, model, device):
    xs, ys = [], []
    for x, y in tqdm(dataloader, leave=False):
        with torch.no_grad():
            xs.append(model(x.to(device)).detach().cpu().numpy())
        ys.append(y.numpy())
    xs = np.vstack(xs)
    ys = np.hstack(ys)
    return torch.tensor(xs), torch.tensor(ys)

In [ ]:
class PrecomputedDataset(Dataset):
    def __init__(self, features, labels):
        super(PrecomputedDataset, self).__init__()
        self.features = features
        self.labels = labels.unsqueeze(-1)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx].float()

In [ ]:
preprocessing = transforms.Resize((98, 98))
train_dataset = BaselineDataset(TRAIN_IMAGES_PATH, preprocessing, "train")
val_dataset = BaselineDataset(VAL_IMAGES_PATH, preprocessing, "train")

In [ ]:
train_dataloader = DataLoader(train_dataset, shuffle=True, batch_size=BATCH_SIZE)
val_dataloader = DataLoader(val_dataset, shuffle=False, batch_size=BATCH_SIZE)

### 2.2. Building the models and precomputing the features

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Working on {device}.")

In [ ]:
feature_extractor = torch.hub.load("facebookresearch/dinov2", "dinov2_vits14").to(
    device
)
feature_extractor.eval()
linear_probing = torch.nn.Sequential(
    torch.nn.Linear(feature_extractor.num_features, 1), torch.nn.Sigmoid()
).to(device)

In [ ]:
train_dataset = PrecomputedDataset(
    *precompute(train_dataloader, feature_extractor, device)
)
val_dataset = PrecomputedDataset(*precompute(val_dataloader, feature_extractor, device))

In [ ]:
train_dataloader = DataLoader(train_dataset, shuffle=True, batch_size=BATCH_SIZE)
val_dataloader = DataLoader(val_dataset, shuffle=False, batch_size=BATCH_SIZE)

## 3. Training the model

In [ ]:
OPTIMIZER = "Adam"
OPTIMIZER_PARAMS = {"lr": 0.001}
LOSS = "BCELoss"
METRIC = "Accuracy"
NUM_EPOCHS = 100
PATIENCE = 10

In [ ]:
optimizer = getattr(torch.optim, OPTIMIZER)(
    linear_probing.parameters(), **OPTIMIZER_PARAMS
)
criterion = getattr(torch.nn, LOSS)()
metric = getattr(torchmetrics, METRIC)("binary")
min_loss, best_epoch = float("inf"), 0

In [ ]:
for epoch in range(NUM_EPOCHS):
    linear_probing.train()
    train_metrics, train_losses = [], []
    for train_x, train_y in tqdm(train_dataloader, leave=False):
        optimizer.zero_grad()
        train_pred = linear_probing(train_x.to(device))
        loss = criterion(train_pred, train_y.to(device))
        loss.backward()
        optimizer.step()
        train_losses.extend([loss.item()] * len(train_y))
        train_metric = metric(train_pred.cpu(), train_y.int().cpu())
        train_metrics.extend([train_metric.item()] * len(train_y))
    print(
        f"Epoch train [{epoch + 1}/{NUM_EPOCHS}] | Loss {np.mean(train_losses):.4f} | Metric {np.mean(train_metrics):.4f}"
    )

    linear_probing.eval()
    val_metrics, val_losses = [], []
    for val_x, val_y in tqdm(val_dataloader, leave=False):
        with torch.no_grad():
            val_pred = linear_probing(val_x.to(device))
        loss = criterion(val_pred, val_y.to(device))
        val_losses.extend([loss.item()] * len(val_y))
        val_metric = metric(val_pred.cpu(), val_y.int().cpu())
        val_metrics.extend([val_metric.item()] * len(val_y))
    print(
        f"Epoch valid [{epoch + 1}/{NUM_EPOCHS}] | Loss {np.mean(val_losses):.4f} | Metric {np.mean(val_metrics):.4f}"
    )

    if np.mean(val_losses) < min_loss:
        mean_val_loss = np.mean(val_losses)
        print(f"New best loss {min_loss:.4f} -> {mean_val_loss:.4f}")
        min_loss = mean_val_loss
        best_epoch = epoch
        torch.save(linear_probing.state_dict(), "best_model.pth")

    if epoch - best_epoch == PATIENCE:
        break

## 4. Making the final prediction

To create a solutions file, you need to generate a CSV with 2 columns.
- **ID**: containing the ID of the image
- **Pred**: with the predicted class (**threshold the prediction to get either 0 or 1**)

In [ ]:
linear_probing.load_state_dict(torch.load("best_model.pth", weights_only=True))
linear_probing.eval()
linear_probing.to(device)
prediction_dict = {}

In [ ]:
with h5py.File(TEST_IMAGES_PATH, "r") as hdf:
    test_ids = list(hdf.keys())

In [ ]:
solutions_data = {"ID": [], "Pred": []}
with h5py.File(TEST_IMAGES_PATH, "r") as hdf:
    for test_id in tqdm(test_ids):
        img = (
            preprocessing(torch.tensor(np.array(hdf.get(test_id).get("img"))))
            .unsqueeze(0)
            .float()
        )
        pred = linear_probing(feature_extractor(img.to(device))).detach().cpu()
        solutions_data["ID"].append(int(test_id))
        solutions_data["Pred"].append(int(pred.item() > 0.5))
solutions_data = pd.DataFrame(solutions_data).set_index("ID")
solutions_data.to_csv("baseline.csv")